# Credit Card Fraud Detection with Amazon SageMaker

## Overview

Production-ready fraud detection using XGBoost on AWS SageMaker.

### What You'll Build

1. Download and prepare fraud detection dataset (284K transactions)
2. Handle severe class imbalance using SMOTE
3. Train XGBoost model on SageMaker
4. Deploy to real-time endpoint
5. Make predictions and integrate with applications

### Estimated Costs

- Training: $0.10-0.25 (ml.m5.xlarge, 5-10 min)
- Endpoint: $0.10/hour (ml.t2.medium)
- S3: < $0.01
- **Total**: < $1.00

---

## Step 0: Fix SageMaker Installation (Run This First!)

**IMPORTANT**: Run this cell first to ensure SageMaker SDK is properly installed.

In [ ]:
import sys
import subprocess

print("Checking SageMaker installation...\n")

# Force reinstall SageMaker to fix any issues
try:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', 
        '--upgrade', '--no-cache-dir',
        'sagemaker==2.200.0',
        'scikit-learn==1.4.0',
        'imbalanced-learn==0.12.0'
    ], stdout=subprocess.DEVNULL)
    print("✓ SageMaker SDK installed/upgraded successfully")
except Exception as e:
    print(f"Note: {e}")

print("\n⚠️  RESTART KERNEL NOW: Kernel → Restart Kernel")
print("Then proceed to Step 1")

## Step 1: Import Libraries

In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
import json
import os
import sys
from datetime import datetime
import time

# AWS SDK
import boto3
import sagemaker

# Machine Learning
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Add project source to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(project_root, 'src'))

from data_processor import FraudDataProcessor

print("✓ All libraries imported successfully!")
print(f"✓ Python: {sys.version.split()[0]}")
print(f"✓ Pandas: {pd.__version__}")
print(f"✓ NumPy: {np.__version__}")
print(f"✓ boto3: {boto3.__version__}")

## Step 2: Configure AWS & SageMaker

In [ ]:
# Initialize SageMaker (with fallback for broken installations)
try:
    sagemaker_session = sagemaker.Session()
    print("✓ SageMaker Session created")
except:
    # Fallback: create manually
    from sagemaker.session import Session
    sagemaker_session = Session(boto_session=boto3.Session())
    print("✓ SageMaker Session created (fallback)")

region = sagemaker_session.boto_region_name
role = sagemaker.get_execution_role()

# Get account ID
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']

# Create unique bucket
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
bucket_name = f"fraud-detection-{account_id}-{timestamp}"

# Create S3 bucket
s3 = boto3.client('s3', region_name=region)
try:
    if region == 'us-east-1':
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={'LocationConstraint': region}
        )
    print(f"✓ Created bucket: {bucket_name}")
except Exception as e:
    print(f"Note: {e}")

# S3 paths
prefix = 'fraud-detection'
train_s3 = f's3://{bucket_name}/{prefix}/train'
val_s3 = f's3://{bucket_name}/{prefix}/validation'
output_s3 = f's3://{bucket_name}/{prefix}/output'

print(f"\n{'='*60}")
print(f"Region: {region}")
print(f"Bucket: {bucket_name}")
print(f"Role: {role.split('/')[-1]}")
print(f"{'='*60}")

## Step 3: Download Dataset

In [ ]:
# Initialize processor
processor = FraudDataProcessor(random_state=42)

# Create data directory
data_dir = os.path.join(project_root, 'data')
os.makedirs(data_dir, exist_ok=True)

# Download dataset
dataset_path = os.path.join(data_dir, 'creditcard.csv')

if not os.path.exists(dataset_path):
    print("Downloading dataset (1-2 minutes)...")
    processor.download_dataset(
        url="https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv",
        local_path=dataset_path
    )
else:
    print(f"✓ Dataset exists: {dataset_path}")

# Load dataset
df = processor.load_dataset(dataset_path)

print("\nDataset Preview:")
display(df.head())

print("\nDataset Info:")
display(df.describe())

## Step 4: Analyze Class Imbalance

In [ ]:
# Analyze imbalance
stats = processor.analyze_imbalance(df)

if stats['fraud_percentage'] < 1.0:
    print("\n⚠️  SEVERE CLASS IMBALANCE!")
    print("   Using SMOTE to balance...")

## Step 5: Apply SMOTE Balancing

In [ ]:
# Apply SMOTE
df_balanced = processor.balance_dataset(df, sampling_strategy=0.3)

# Analyze results
balanced_stats = processor.analyze_imbalance(df_balanced)

print(f"\nOriginal: {len(df):,} transactions")
print(f"Balanced: {len(df_balanced):,} transactions")
print(f"New fraud ratio: {balanced_stats['fraud_percentage']:.1f}%")

## Step 6: Split & Upload Data

**IMPORTANT**: XGBoost requires target column FIRST in CSV format.

In [ ]:
# Split data
X = df_balanced.drop('Class', axis=1)
y = df_balanced['Class']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ⚠️ CRITICAL: Put target FIRST for XGBoost
train_df = pd.concat([y_train, X_train], axis=1)
val_df = pd.concat([y_val, X_val], axis=1)

print(f"Train: {len(train_df):,} | Fraud: {sum(y_train==1):,}")
print(f"Val: {len(val_df):,} | Fraud: {sum(y_val==1):,}")

# Save locally (no header, target first)
train_path = os.path.join(data_dir, 'train.csv')
val_path = os.path.join(data_dir, 'validation.csv')

train_df.to_csv(train_path, index=False, header=False)
val_df.to_csv(val_path, index=False, header=False)

print(f"\n✓ Saved: {train_path}")
print(f"✓ Saved: {val_path}")

# Verify format
print(f"\n✓ Target (Class) is in FIRST column")
print(f"✓ Train shape: {train_df.shape}")
print(f"✓ First column values: {train_df.iloc[:5, 0].tolist()}")

In [ ]:
# Upload to S3
print("Uploading to S3...\n")

train_s3_path = sagemaker_session.upload_data(
    path=train_path,
    bucket=bucket_name,
    key_prefix=f"{prefix}/train"
)

val_s3_path = sagemaker_session.upload_data(
    path=val_path,
    bucket=bucket_name,
    key_prefix=f"{prefix}/validation"
)

print(f"✓ Train: {train_s3_path}")
print(f"✓ Val: {val_s3_path}")

## Step 7: Configure Training

In [ ]:
# Hyperparameters
hyperparameters = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'num_round': 100,
    'max_depth': 6,
    'eta': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8
}

print("Hyperparameters:")
for k, v in hyperparameters.items():
    print(f"  {k}: {v}")

## Step 8: Create Estimator

In [ ]:
# Get XGBoost container
from sagemaker import image_uris

container = image_uris.retrieve(
    framework='xgboost',
    region=region,
    version='1.5-1'
)

print(f"Container: {container}")

# Create estimator
from sagemaker.estimator import Estimator

xgb = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=output_s3,
    hyperparameters=hyperparameters,
    sagemaker_session=sagemaker_session
)

print("\n✓ Estimator created")

## Step 9: Train Model

**This takes 5-10 minutes**

In [ ]:
# Training inputs
train_input = sagemaker.inputs.TrainingInput(
    s3_data=train_s3_path,
    content_type='text/csv'
)

val_input = sagemaker.inputs.TrainingInput(
    s3_data=val_s3_path,
    content_type='text/csv'
)

# Start training
print("Starting training (5-10 minutes)...\n")
start = time.time()

xgb.fit(
    inputs={'train': train_input, 'validation': val_input},
    wait=True,
    logs='All'
)

elapsed = (time.time() - start) / 60
print(f"\n✓ Training complete! ({elapsed:.1f} min)")
print(f"Model: {xgb.model_data}")

## Step 10: Deploy Endpoint

**Deployment: 5-8 minutes**  
**Cost: ~$0.10/hour while running**

⚠️ **Remember to delete when done!**

In [ ]:
# Deploy
endpoint_name = f"fraud-detection-{timestamp}"

print(f"Deploying to: {endpoint_name}")
print("This takes 5-8 minutes...\n")

start = time.time()

predictor = xgb.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    endpoint_name=endpoint_name
)

elapsed = (time.time() - start) / 60
print(f"\n✓ Deployed! ({elapsed:.1f} min)")
print(f"Endpoint: {endpoint_name}")

## Step 11: Test Endpoint

In [ ]:
# Setup predictor
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

predictor.serializer = CSVSerializer()
predictor.deserializer = CSVDeserializer()

# Test legitimate transaction
print("Testing LEGITIMATE transaction...")
legit = val_df[val_df.iloc[:, 0] == 0].iloc[0].iloc[1:].values.reshape(1, -1)
result = predictor.predict(legit)
print(f"Prediction: {result[0][0]} (0=legit, 1=fraud)")
print(f"Expected: 0 (legitimate)")

# Test fraudulent transaction
print("\nTesting FRAUDULENT transaction...")
fraud = val_df[val_df.iloc[:, 0] == 1].iloc[0].iloc[1:].values.reshape(1, -1)
result = predictor.predict(fraud)
print(f"Prediction: {result[0][0]} (0=legit, 1=fraud)")
print(f"Expected: 1 (fraud)")

print("\n✓ Testing complete!")

## Step 12: Production Integration

In [ ]:
# Production example using boto3
runtime = boto3.client('sagemaker-runtime', region_name=region)

# Test transaction (skip first column which is target)
test_data = val_df.iloc[0].iloc[1:].values.tolist()
csv_data = ','.join(map(str, test_data))

# Invoke endpoint
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='text/csv',
    Body=csv_data
)

result = response['Body'].read().decode()
print(f"Result: {result}")

print("\n" + "="*60)
print("Integration Code:")
print("="*60)
print(f'''
import boto3

runtime = boto3.client('sagemaker-runtime', region_name='{region}')
csv_data = ','.join(map(str, transaction_features))

response = runtime.invoke_endpoint(
    EndpointName='{endpoint_name}',
    ContentType='text/csv',
    Body=csv_data
)

prediction = float(response['Body'].read().decode())
is_fraud = prediction >= 0.5
''')
print("="*60)

## Step 13: Monitor Endpoint

In [ ]:
# Endpoint info
sm = boto3.client('sagemaker', region_name=region)
info = sm.describe_endpoint(EndpointName=endpoint_name)

print(f"Endpoint: {info['EndpointName']}")
print(f"Status: {info['EndpointStatus']}")
print(f"Created: {info['CreationTime']}")

# Calculate cost
uptime = datetime.now(info['CreationTime'].tzinfo) - info['CreationTime']
hours = uptime.total_seconds() / 3600
cost = hours * 0.10

print(f"\nUptime: {hours:.2f} hours")
print(f"Cost: ${cost:.4f}")
print("\n⚠️  Delete endpoint to stop charges!")

## Step 14: Cleanup

**⚠️ RUN THIS TO STOP CHARGES!**

In [ ]:
# Delete endpoint
print("Deleting endpoint...\n")

try:
    predictor.delete_endpoint(delete_endpoint_config=True)
    print("✓ Endpoint deleted")
    print("✓ Charges stopped")
except Exception as e:
    print(f"Error: {e}")
    print(f"Manually delete: {endpoint_name}")

In [ ]:
# Optional: Delete S3 bucket
DELETE_S3 = False  # Set True to delete

if DELETE_S3:
    print(f"Deleting S3: {bucket_name}...")
    s3_res = boto3.resource('s3')
    bucket = s3_res.Bucket(bucket_name)
    bucket.objects.all().delete()
    bucket.delete()
    print("✓ S3 deleted")
else:
    print(f"S3 cleanup skipped")
    print(f"To delete: aws s3 rb s3://{bucket_name} --force")

## Summary

### ✓ Completed

- Downloaded fraud dataset (284K transactions)
- Applied SMOTE balancing (0.17% → 30% fraud)
- Trained XGBoost on SageMaker
- Deployed to real-time endpoint
- Tested predictions
- Cleaned up resources

### Key Fixes Applied

1. **SageMaker installation fix** in Step 0
2. **Target column first** for XGBoost CSV format
3. **Fallback Session creation** for broken installations
4. **Proper test data handling** (skip target column)

### Next Steps

1. Hyperparameter tuning
2. Model monitoring with CloudWatch
3. A/B testing multiple models
4. Batch predictions
5. MLOps with SageMaker Pipelines

---

**Workshop Complete!** 🎉
